# Qualitatively Evaluate Models
Pull trained model weights down from s3, load, and infer

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../src'))

In [ ]:
!pip install -r DOE-LLNL-ML-ModelDev/src/requirements.txt

In [ ]:
from llnl_ml.model.unet import UNet
from llnl_ml.lightning import SegmenationLightningModule

import torch

from llnl_ml.data import get_data_loaders

import matplotlib.pyplot as plt

In [ ]:
image_folder = "Dataset-2K/images/"
mask_folder = "Dataset-2K/masks/"
batch_size = 4
num_workers = 2
image_mode = "L"

n_batches = 1

In [ ]:
# Downloaded the model.tar.gz file from S3 and upacked in terminal
# aws s3 cp s3://path/to/model.tar.gz ./
# tar -xf model.tar.gz
model_dir = "models/segmentation-1698182607-032f"
model_name = "epoch=48-val_loss=0.15.ckpt"
checkpoint_path = f"{model_dir}/{model_name}"

In [ ]:
# By default, pytorch lightning will save the state_dict of the Lightning Module
# Create the model and PL Module wrapping it, load the state_dict into the PL module which will
# also load the underying model inside.
pl_model = SegmenationLightningModule.load_from_checkpoint(checkpoint_path)
model = pl_model.model
model.eval()

In [ ]:
# Get train, val and test loaders
train_loader, val_loader, test_loader = get_data_loaders(
    image_folder=image_folder,
    mask_folder=mask_folder,
    batch_size=batch_size,
    num_workers=num_workers,
    image_mode=image_mode,
    image_size=2048,
    use_random_resize=True,
    use_random_rotate=False,
    color_jitter=0.0,
    center_crop=False,
)

In [ ]:
def plot_batch_results(images, masks, results, fname=None):
    n_plots = images.shape[0]
    fig, axs = plt.subplots(n_plots, 4, figsize=(12,12))
    # Add titles
    axs[0, 0].set_title("Input Image")
    axs[0, 1].set_title("Mask")
    axs[0, 2].set_title("Raw Output")
    axs[0, 3].set_title("Thresholded Output")
    for row in range(images.shape[0]):
        axs[row, 0].imshow(images[row], cmap="gray")
        axs[row, 0].set_axis_off()
        axs[row, 1].imshow(masks[row])
        axs[row, 1].set_axis_off()
        axs[row, 2].imshow(results[row])
        axs[row, 2].set_axis_off()
        axs[row, 3].imshow(results[row] > 0)
        axs[row, 3].set_axis_off()
    fig.subplots_adjust(wspace=0, hspace=0)
    if fname:
        plt.savefig(fname, bbox_inches="tight")

In [ ]:
# For simple testing, grab the first batch of test and train so we 
# can see the performance difference
loss_fn = torch.nn.BCEWithLogitsLoss()
for batch_count, test_batch in enumerate(test_loader):
    if batch_count >= n_batches:
        break
    with torch.no_grad():
        output = model(test_batch[0])
        result = output.detach().squeeze().numpy()
        images = test_batch[0].squeeze().numpy()
        masks = test_batch[1]['masks'].squeeze().numpy()
    filename = f"{model_dir}/test_{images.shape[-1]}_batch{batch_count}.png"
    plot_batch_results(images, masks, result, fname=filename)
    print(f"Batch {batch_count}: {loss_fn(output.squeeze(), test_batch[1]['masks']) = }")

In [ ]:
for batch_count, train_batch in enumerate(train_loader):
    if batch_count >= n_batches:
        break
    with torch.no_grad():
        output = model(train_batch[0])
        result = output.detach().squeeze().numpy()
        images = train_batch[0].squeeze().numpy()
        masks = train_batch[1]['masks'].squeeze().numpy()
    filename = f"{model_dir}/train_{images.shape[-1]}_batch{batch_count}.png"
    plot_batch_results(images, masks, result, fname=filename)
    print(f"Batch {batch_count}: {loss_fn(output.squeeze(), train_batch[1]['masks']) = }")